# Four integrators on one Gravitas system

A reviewed notebook for the Gravitas compute page. It reads one `gravitas.orbital-system/1`, the file the Orbital System Builder saves, handed to it as the text `gravitas_input`, and its two settings as the text `gravitas_params`. It uses only the file's `initial` block: the gravitational constant and each body's position, velocity and mass, in simulation units.

It integrates the same Newtonian point-mass system four ways from that start: explicit (forward) Euler, symplectic Euler (the scheme Gravitas uses by default: velocity first, then position with the new velocity), velocity Verlet (the leapfrog form) and fourth-order Runge-Kutta. All four use one fixed step. The step is the period of the system's tightest pair, taken as a circular orbit at that pair's separation, divided by the setting `steps`. The run lasts `orbits` periods of the system's widest orbit, taken the same way about the barycenter. Both are read from the initial state, so any file the Builder saves has them.

The number that is returned for each method is the energy drift at the end of the run, in percent, defined as Gravitas's own readout defines it: 100 times the total energy now minus the total energy at the start, over the magnitude of the start's. Kinetic energy plus the pairwise potential, the same sums. The Energy tab of Gravitas shows the same figure for the same system, which is the comparison this notebook is for. It also returns the whole drift curve of each method as a `table` and a log plot as a `figure`.

What it does not do: it has no softening, no merging, no collisions and no post-Newtonian term, which the application has as options. A method's drift depends on the step, so a number here is the drift at this step, not a property of the method alone.

In [ ]:
# Read the input and the two settings.
import json, hashlib
import numpy as np

doc = json.loads(gravitas_input)
params = json.loads(gravitas_params)
assert doc["format"] == "gravitas.orbital-system" and doc["version"] == 1, "this notebook reads gravitas.orbital-system/1"
assert isinstance(doc.get("initial"), dict), "the file has no initial state: save it again from the Orbital System Builder"
ORBITS = float(params.get("orbits", 3))
PER = float(params.get("steps", 500))
assert 1 <= ORBITS <= 10, "orbits is between 1 and 10"
assert 50 <= PER <= 1000, "steps is between 50 and 1000"
input_digest = hashlib.sha256(gravitas_input.encode("utf-8")).hexdigest()

G = float(doc["initial"]["G"])
bodies = doc["initial"]["bodies"]
names = [b.get("name") or f"body {i + 1}" for i, b in enumerate(doc["bodies"])]
mass = np.array([b["mass"] for b in bodies], float)
x0 = np.array([[b["x"], b["y"]] for b in bodies], float)
v0 = np.array([[b["vx"], b["vy"]] for b in bodies], float)
N = len(mass)
assert 2 <= N <= 12 and len(names) == N, "the file's initial state does not match its bodies"
print(f"{N} bodies, G = {G:g}: " + ", ".join(names))

In [ ]:
# The step and the length of the run, from the initial state alone.
d0 = np.linalg.norm(x0[:, None, :] - x0[None, :, :], axis=2)
iu = np.triu_indices(N, 1)
k = int(np.argmin(d0[iu]))
i, j = iu[0][k], iu[1][k]
t_short = 2 * np.pi * np.sqrt(d0[i, j] ** 3 / (G * (mass[i] + mass[j])))
com = (mass[:, None] * x0).sum(axis=0) / mass.sum()
r_far = float(np.max(np.linalg.norm(x0 - com, axis=1)))
t_long = 2 * np.pi * np.sqrt(r_far ** 3 / (G * mass.sum()))
dt = t_short / PER
steps = int(np.ceil(ORBITS * t_long / dt))
assert steps <= 60000, "that is more than 60000 steps: lower the orbits or the steps"
print(f"tightest pair {names[i]} and {names[j]}, period {t_short:.4g}; widest orbit period {t_long:.4g}; step {dt:.4g}; {steps} steps")

In [ ]:
# Gravity, energy, and the four integrators.
def accel(x):
    d = x[None, :, :] - x[:, None, :]
    r2 = (d ** 2).sum(axis=2)
    np.fill_diagonal(r2, 1.0)
    w = r2 ** -1.5
    np.fill_diagonal(w, 0.0)
    return G * ((mass[None, :] * w)[:, :, None] * d).sum(axis=1)

def energy(x, v):
    kinetic = 0.5 * float((mass * (v ** 2).sum(axis=1)).sum())
    r = np.sqrt((((x[:, None, :] - x[None, :, :]) ** 2).sum(axis=2)))
    np.fill_diagonal(r, np.inf)
    potential = -0.5 * G * float(((mass[:, None] * mass[None, :]) / r).sum())
    return kinetic + potential

def euler(x, v, h):
    return x + v * h, v + accel(x) * h

def symplectic(x, v, h):
    v = v + accel(x) * h
    return x + v * h, v

def verlet(x, v, h):
    v = v + 0.5 * h * accel(x)
    x = x + h * v
    return x, v + 0.5 * h * accel(x)

def rk4(x, v, h):
    k1x, k1v = v, accel(x)
    k2x, k2v = v + 0.5 * h * k1v, accel(x + 0.5 * h * k1x)
    k3x, k3v = v + 0.5 * h * k2v, accel(x + 0.5 * h * k2x)
    k4x, k4v = v + h * k3v, accel(x + h * k3x)
    return (x + h / 6 * (k1x + 2 * k2x + 2 * k3x + k4x),
            v + h / 6 * (k1v + 2 * k2v + 2 * k3v + k4v))

METHODS = [("euler", "explicit Euler", euler), ("symplectic", "symplectic Euler (Gravitas default)", symplectic),
           ("verlet", "velocity Verlet", verlet), ("rk4", "Runge-Kutta 4", rk4)]
E0 = energy(x0, v0)
assert E0 < 0, "the system is not bound as a whole: its total energy is not negative, so a percent drift has no meaning here"
stride = max(1, steps // 240)
times = np.arange(0, steps + 1, stride) * dt
curves = {}
for key, label, step in METHODS:
    x, v = x0.copy(), v0.copy()
    out = [0.0]
    for s in range(1, steps + 1):
        x, v = step(x, v, dt)
        if s % stride == 0:
            out.append(100.0 * (energy(x, v) - E0) / abs(E0))
    if (steps % stride) != 0:
        out.append(100.0 * (energy(x, v) - E0) / abs(E0))
    curves[key] = (np.array(out), 100.0 * (energy(x, v) - E0) / abs(E0))
    print(f"{label}: drift {curves[key][1]:+.4g} %")
if len(times) != len(curves["euler"][0]):
    times = np.append(times, steps * dt)

In [ ]:
# What the notebook returns: an envelope, a table and a figure.
LICENSE = {"status": "generated", "statement": "Computed by Gravitas from the file it was given; the notebook is MIT, and the file keeps whatever terms its author gave it.", "basis": "the file is the reader's own or the example Gravitas wrote"}
def quantity(key, value):
    return {"id": f"drift.{key}", "value": float(value), "unit": "%", "origin": "derived",
            "uncertainty": {"kind": "none"}}

envelope = {
    "format": "gravitas.artifact", "formatVersion": 1,
    "id": f"compute:integrators-compared:{input_digest[:16]}",
    "made": {"app": "gravitas"},
    "source": {"kind": "analysis", "id": "compute/integrators-compared", "version": "1", "digest": input_digest},
    "provenance": {
        "credit": "Gravitas Orbital System Builder file, integrated by the reviewed notebook integrators-compared",
        "license": LICENSE,
        "retrieved": "2026-10-11",
        "citations": ["Hairer, Lubich and Wanner 2006, Geometric Numerical Integration, 2nd ed., Springer, chapter I"],
        "reductions": [
            f"{N} point masses, Newtonian gravity at G = {G:g}, from the file's initial state; no softening.",
            f"Fixed step {dt:.6g} = tightest-pair period {t_short:.6g} over {PER:g}; {steps} steps = {ORBITS:g} periods of the widest orbit ({t_long:.6g}).",
            "Drift is 100 (E - E0) / |E0| at the end of the run, E kinetic plus pairwise potential energy.",
        ],
    },
    "quantities": [quantity(k, curves[k][1]) for k, _, _ in METHODS] + [
        {"id": "steps", "value": float(steps), "unit": "count", "origin": "derived", "uncertainty": {"kind": "none"}},
    ],
    "warnings": [
        "a drift is the drift at this step: halve the step and a method of order p changes by about 2**p, which is what to check before comparing methods",
        "the energy of a chaotic or close-encounter system can jump at one step; the final value then says little about the rest of the run, so read the table too",
        "Gravitas's own readout includes whatever softening, merging and relativity the scenario has switched on; this notebook has none of them",
    ],
}

columns = [{"id": "time", "name": "time", "unit": "sim", "role": "x", "values": [float(v) for v in times]}]
for key, label, _ in METHODS:
    columns.append({"id": f"drift-{key}", "name": f"energy drift, {label}", "unit": "%", "role": "value",
                    "values": [float(v) for v in curves[key][0]]})
table = {
    "format": "gravitas.observation", "formatVersion": 1, "kind": "table",
    "id": f"compute:integrators-compared:{input_digest[:16]}:drift", "title": "Energy drift of four integrators on one system",
    "object": {"name": ", ".join(names)}, "facility": {"name": "Gravitas compute page"},
    "origin": "model",
    "source": {"kind": "import", "id": "gravitas.orbital-system", "version": "1"},
    "credit": envelope["provenance"]["credit"], "license": LICENSE, "retrieved": "2026-10-11",
    "citations": [{"text": envelope["provenance"]["citations"][0]}],
    "reductions": envelope["provenance"]["reductions"],
    "axes": {"x": "time", "y": "drift-symplectic"},
    "columns": columns, "masks": [], "annotations": [],
}

import io, base64
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.5), dpi=100)
for key, label, _ in METHODS:
    ax.semilogy(times, np.maximum(np.abs(curves[key][0]), 1e-12), label=label)
ax.set_xlabel("time (simulation units)")
ax.set_ylabel("|energy drift| (%)")
ax.legend(fontsize=6)
ax.set_title(f"{', '.join(names)}: {steps} steps of {dt:.3g}", fontsize=8)
fig.tight_layout()
png = io.BytesIO()
# No date or version in the file, so the same run gives the same bytes.
fig.savefig(png, format="png", metadata={"Software": None})
plt.close(fig)

gravitas_output = {
    "envelope": envelope,
    "table": table,
    "figure": {"type": "image/png", "base64": base64.b64encode(png.getvalue()).decode("ascii")},
}
print(len(times), "rows returned")